In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
# 1. Import Libraries
import numpy as np
import pandas as pd

In [3]:
# 2. Load Data
train= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test= pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [4]:
#column with highest missing value
print(train.isnull().sum().sort_values(ascending=False).head())

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
dtype: int64


In [5]:
# 3. Drop Columns
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']

train = train.drop(cols_to_drop, axis=1)
test = test.drop(cols_to_drop, axis=1)

In [6]:
# 4. Separate Features & Target
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

In [7]:
#5. Missing Value Handling
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Numeric -> Median
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(X[num_cols].median())

# Categorical -> Missing
X[cat_cols] = X[cat_cols].fillna('Missing')
test[cat_cols] = test[cat_cols].fillna('Missing')

#Verify missing value
print(X.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [8]:
#6.One-Hot Encode
X = pd.get_dummies(X)
test_encoded = pd.get_dummies(test)

X, test_encoded = X.align(
    test_encoded,
    join='left',
    axis=1,
    fill_value=0
)

print(X.shape)
print(test_encoded.shape)

(1460, 287)
(1459, 287)


In [9]:
#7. Log Transform Target
y= np.log1p(y)
print(y.head())

0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [10]:
#8. Train Model
from sklearn.ensemble import RandomForestRegressor

model=RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

In [11]:
#9.Create the Search Space

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

search = RandomizedSearchCV(
    estimator = model,
    param_distributions = param_dist,
    n_iter = 10,
    cv = 5 ,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
    
)

In [12]:
#10.Start the Search
search.fit(X,y)

#View the Best Parameters
print("Best Parameters:")
print(search.best_params_)

#Best Cross-Validation Score
print("\nBest RMSE:")
print(-search.best_score_)

#Best Model
best_model = search.best_estimator_

Best Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 10}

Best RMSE:
0.14361987297314033


In [13]:
# 11. Train on Full Data
best_model.fit(X, y)

RandomForestRegressor(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=300, n_jobs=-1, random_state=42)

In [14]:
# 12. Predict Test Data
prediction = best_model.predict(test_encoded)


#Convert Back to Real Prices
prediction = np.expm1(prediction)
print(prediction)

print(prediction[:10])

[124764.12661601 153012.78162079 179053.0221235  ... 152179.52369627
 114364.94345842 235166.52198431]
[124764.12661601 153012.78162079 179053.0221235  180166.31750803
 194757.84257637 182085.95920612 164224.42059908 176283.67761944
 187121.81652345 122006.37776241]


In [15]:
#13. Submission File
submission=pd.DataFrame({
    'Id':test['Id'],
    'SalePrice': prediction
})

submission.to_csv('submission_v7.csv',index=False)

submission.head()

,Id,SalePrice
0,1461,124764.126616
1,1462,153012.781621
2,1463,179053.022123
3,1464,180166.317508
4,1465,194757.842576
